# IndoBERT Masked Language Modeling - Target Domain Adaptation

This notebook trains IndoBERT using Masked Language Modeling (MLM) on the target domain data (reviewContent from target_final.csv)

In [1]:
# !pip install accelerate==0.28.0
# !pip install matplotlib==3.10.9

## Import Libraries

In [2]:
import pandas as pd
import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)
from datasets import Dataset
import numpy as np

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Load Target Data

In [3]:
# Load the target_final.csv file
df = pd.read_csv('../../datasets/lazada_train.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

Dataset shape: (8187, 3)

Columns: ['reviewContent', 'label', 'rating']

First few rows:


,reviewContent,label,rating
0,2 star saja,negative,2
1,pkok'ya puas banget belanja di java elektronik...,positive,5
2,"bareng cepat sampenya, packing a rapi, tv a ke...",negative,3
3,"Mantap, gambar jernih kualitas bagus. Pengirim...",positive,5
4,Pngiriman cepat thnks lazada,positive,5


In [4]:
# Extract text data (use 'content' column based on preprocessing notebook)
texts = df['reviewContent'].dropna().tolist()

print(f"Total texts for MLM training: {len(texts)}")
print(f"\nSample text:")
print(texts[0])

Total texts for MLM training: 8187

Sample text:
2 star saja


## Load IndoBERT Model and Tokenizer

In [5]:
# Load IndoBERT tokenizer and model
model_name = "indolem/indobert-base-uncased"

print(f"Loading tokenizer and model: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)

print(f"Model loaded successfully!")
print(f"Model parameters: {model.num_parameters():,}")

Loading tokenizer and model: indolem/indobert-base-uncased


c:\Anaconda3\envs\computer_vision\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at indolem/indobert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Model loaded successfully!
Model parameters: 110,591,667


## Prepare Data for MLM Training

In [6]:
# Create dataset from texts
dataset_dict = {"text": texts}
dataset = Dataset.from_dict(dataset_dict)

print(f"Dataset size: {len(dataset)}")
print(f"\nDataset features: {dataset.features}")
print(f"\nFirst example:")
print(dataset[0])

Dataset size: 8187

Dataset features: {'text': Value(dtype='string', id=None)}

First example:
{'text': '2 star saja'}


In [7]:
# Tokenize the dataset
def tokenize_function(examples):
    # Tokenize with truncation and padding
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length",
        return_special_tokens_mask=True
    )

# Apply tokenization
print("Tokenizing dataset...")
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing"
)

print(f"Tokenization complete!")
print(f"Tokenized dataset features: {tokenized_dataset.features}")

Tokenizing dataset...


Tokenizing:   0%|          | 0/8187 [00:00<?, ? examples/s]

Tokenization complete!
Tokenized dataset features: {'input_ids': Sequence(feature=Value(dtype='int32', id=None), length=-1, id=None), 'token_type_ids': Sequence(feature=Value(dtype='int8', id=None), length=-1, id=None), 'attention_mask': Sequence(feature=Value(dtype='int8', id=None), length=-1, id=None), 'special_tokens_mask': Sequence(feature=Value(dtype='int8', id=None), length=-1, id=None)}


In [8]:
# Split dataset into train and validation (90/10 split)
train_test_split = tokenized_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = train_test_split['train']
eval_dataset = train_test_split['test']

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(eval_dataset)}")

Training samples: 7368
Validation samples: 819


In [9]:
# Create data collator for MLM
# mlm_probability=0.15 means 15% of tokens will be masked
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

print("Data collator created for MLM with 15% masking probability")

Data collator created for MLM with 15% masking probability


## Configure Training Parameters

In [10]:
# Define training arguments
training_args = TrainingArguments(
    output_dir="./indobert_mlm_target",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    save_steps=500,
    save_total_limit=2,
    evaluation_strategy="steps",
    eval_steps=500,
    logging_steps=100,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=500,
    fp16=torch.cuda.is_available(),  # Use mixed precision if GPU available
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    push_to_hub=False,
    report_to="none"  # Disable wandb/tensorboard
)

print("Training configuration:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  FP16: {training_args.fp16}")

Training configuration:
  Epochs: 3
  Batch size: 8
  Learning rate: 5e-05
  FP16: True


In [11]:
# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

print("Trainer initialized successfully!")

C:\Users\prk\AppData\Roaming\Python\Python312\site-packages\accelerate\accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(


Trainer initialized successfully!


## Train the Model

In [12]:
# Start training
print("Starting MLM training on target domain data...")
print("=" * 60)

train_result = trainer.train()

print("\n" + "=" * 60)
print("Training completed!")
print(f"Training loss: {train_result.training_loss:.4f}")
print(f"Training time: {train_result.metrics['train_runtime']:.2f} seconds")

Starting MLM training on target domain data...


  0%|          | 0/2763 [00:00<?, ?it/s]

{'loss': 5.2264, 'grad_norm': 58.77362060546875, 'learning_rate': 9.4e-06, 'epoch': 0.11}
{'loss': 4.2223, 'grad_norm': 118.9681167602539, 'learning_rate': 1.93e-05, 'epoch': 0.22}
{'loss': 3.943, 'grad_norm': 67.78063201904297, 'learning_rate': 2.93e-05, 'epoch': 0.33}
{'loss': 3.6022, 'grad_norm': 46.650699615478516, 'learning_rate': 3.9200000000000004e-05, 'epoch': 0.43}
{'loss': 3.7508, 'grad_norm': 76.56702423095703, 'learning_rate': 4.92e-05, 'epoch': 0.54}


  0%|          | 0/103 [00:00<?, ?it/s]

{'eval_loss': 3.1332790851593018, 'eval_runtime': 10.564, 'eval_samples_per_second': 77.528, 'eval_steps_per_second': 9.75, 'epoch': 0.54}
{'loss': 3.5055, 'grad_norm': 28.944664001464844, 'learning_rate': 4.798939460892621e-05, 'epoch': 0.65}
{'loss': 3.2263, 'grad_norm': 27.870777130126953, 'learning_rate': 4.5779938135218736e-05, 'epoch': 0.76}
{'loss': 3.192, 'grad_norm': 27.705196380615234, 'learning_rate': 4.359257622624834e-05, 'epoch': 0.87}
{'loss': 3.0177, 'grad_norm': 26.15093231201172, 'learning_rate': 4.138311975254088e-05, 'epoch': 0.98}
{'loss': 3.0097, 'grad_norm': 23.31129264831543, 'learning_rate': 3.917366327883341e-05, 'epoch': 1.09}


...

  0%|          | 0/103 [00:00<?, ?it/s]

Checkpoint destination directory ./indobert_mlm_target\checkpoint-2500 already exists and is non-empty. Saving will proceed but saved results may be invalid.


{'eval_loss': 2.5272340774536133, 'eval_runtime': 10.5777, 'eval_samples_per_second': 77.427, 'eval_steps_per_second': 9.737, 'epoch': 2.71}
{'loss': 2.4322, 'grad_norm': 22.29029655456543, 'learning_rate': 3.82235969951392e-06, 'epoch': 2.82}
{'loss': 2.6382, 'grad_norm': 22.905126571655273, 'learning_rate': 1.6129032258064516e-06, 'epoch': 2.93}


There were missing keys in the checkpoint model loaded: ['cls.predictions.decoder.weight', 'cls.predictions.decoder.bias'].


{'train_runtime': 948.2172, 'train_samples_per_second': 23.311, 'train_steps_per_second': 2.914, 'train_loss': 3.040904947695765, 'epoch': 3.0}

Training completed!
Training loss: 3.0409
Training time: 948.22 seconds


## Evaluate the Model

In [13]:
# Evaluate on validation set
# NOTE: You MUST run the training cell above first!
# This cell will fail if training hasn't been completed.

print("Evaluating model on validation set...")
eval_results = trainer.evaluate()

print("\nValidation Results:")
print(f"  Validation Loss: {eval_results['eval_loss']:.4f}")
print(f"  Perplexity: {np.exp(eval_results['eval_loss']):.4f}")

Evaluating model on validation set...


  0%|          | 0/103 [00:00<?, ?it/s]


Validation Results:
  Validation Loss: 2.6360
  Perplexity: 13.9578


## Save the Fine-tuned Model

In [14]:
# Save the fine-tuned model and tokenizer
output_dir = "./indobert_mlm_target_final"

print(f"Saving model to {output_dir}...")
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

# Also save model state dict (PyTorch weights only)
weights_path = f"{output_dir}/pytorch_model.bin"
print(f"\nModel weights saved at: {weights_path}")

# Verify saved files
import os
if os.path.exists(output_dir):
    files = os.listdir(output_dir)
    print(f"\nSaved files in {output_dir}:")
    for file in files:
        file_path = os.path.join(output_dir, file)
        if os.path.isfile(file_path):
            size_mb = os.path.getsize(file_path) / (1024 * 1024)
            print(f"  - {file:30s} ({size_mb:.2f} MB)")

print(f"\n✓ Model and tokenizer saved successfully to: {output_dir}")
print("\nYou can now use this model for downstream tasks like sentiment analysis!")
print(f"\nTo load the model later, use:")
print(f"  model = AutoModelForMaskedLM.from_pretrained('{output_dir}')")
print(f"  tokenizer = AutoTokenizer.from_pretrained('{output_dir}')")

Saving model to ./indobert_mlm_target_final...

Model weights saved at: ./indobert_mlm_target_final/pytorch_model.bin

Saved files in ./indobert_mlm_target_final:
  - config.json                    (0.00 MB)
  - generation_config.json         (0.00 MB)
  - model.safetensors              (421.90 MB)
  - special_tokens_map.json        (0.00 MB)
  - tokenizer.json                 (0.70 MB)
  - tokenizer_config.json          (0.00 MB)
  - training_args.bin              (0.00 MB)
  - vocab.txt                      (0.22 MB)

✓ Model and tokenizer saved successfully to: ./indobert_mlm_target_final

You can now use this model for downstream tasks like sentiment analysis!

To load the model later, use:
  model = AutoModelForMaskedLM.from_pretrained('./indobert_mlm_target_final')
  tokenizer = AutoTokenizer.from_pretrained('./indobert_mlm_target_final')


## Training Checkpoints

The model also saves checkpoints during training (every 500 steps). You can find them in the `./indobert_mlm_target` directory.

In [15]:
# Check available checkpoints
checkpoint_dir = "./indobert_mlm_target"

import os
if os.path.exists(checkpoint_dir):
    checkpoints = [d for d in os.listdir(checkpoint_dir) if d.startswith('checkpoint-')]
    checkpoints.sort()
    
    if checkpoints:
        print(f"Available checkpoints in {checkpoint_dir}:")
        for cp in checkpoints:
            cp_path = os.path.join(checkpoint_dir, cp)
            print(f"  - {cp}")
        
        print(f"\nTo load a specific checkpoint:")
        print(f"  model = AutoModelForMaskedLM.from_pretrained('{checkpoint_dir}/checkpoint-XXX')")
    else:
        print("No checkpoints found (training may not have started yet)")
else:
    print(f"Checkpoint directory not found: {checkpoint_dir}")

Available checkpoints in ./indobert_mlm_target:
  - checkpoint-2000
  - checkpoint-2500

To load a specific checkpoint:
  model = AutoModelForMaskedLM.from_pretrained('./indobert_mlm_target/checkpoint-XXX')


## Test the Fine-tuned Model (Optional)

Test the model by predicting masked words

In [16]:
from transformers import pipeline

# Load the fine-tuned model for testing
fill_mask = pipeline(
    "fill-mask",
    model=output_dir,
    tokenizer=output_dir
)

# Test with a sample sentence with [MASK]
test_sentence = "Produk ini sangat [MASK] dan berkualitas."
print(f"Test sentence: {test_sentence}\n")

# Get predictions
predictions = fill_mask(test_sentence, top_k=5)

print("Top 5 predictions:")
for i, pred in enumerate(predictions, 1):
    print(f"{i}. {pred['token_str']:15s} - Score: {pred['score']:.4f}")

RuntimeError: Failed to import transformers.models.bert.modeling_tf_bert because of the following error (look up to see its traceback):
Your currently installed version of Keras is Keras 3, but this is not yet supported in Transformers. Please install the backwards-compatible tf-keras package with `pip install tf-keras`.